# crackSR-bench on Colab

Runtime → Change runtime type → **T4 GPU**. Run the cells from top to bottom.

Datasets are expected on Google Drive in `MyDrive/crackSR-data/` with the sub-folders
`ccic/`, `field/`, `seg/` (see `data/README.md`). Training runs are saved to
`MyDrive/crackSR-runs/` so they survive a disconnect.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Private repo: create a fine-grained GitHub token (read-only, this repo only) and store it in
# Colab "Secrets" (key icon on the left) as GH_TOKEN. Never paste the token into a cell.
from google.colab import userdata
token = userdata.get('GH_TOKEN')
!git clone -q https://{token}@github.com/<your-username>/crackSR-bench.git /content/crackSR-bench
%cd /content/crackSR-bench
!pip install -q -r requirements.txt

In [ ]:
DATA = '/content/drive/MyDrive/crackSR-data'
RUNS = '/content/drive/MyDrive/crackSR-runs'
!mkdir -p $RUNS
!for d in ccic field seg; do [ -e data/$d ] || ln -s $DATA/$d data/$d; done
!ln -sfn $RUNS runs
!ls data runs
!bash scripts/download_weights.sh

## 2. Fixed LR sets (only once; they are written to Drive)

In [ ]:
!python scripts/degrade.py --hr_dir data/seg/val/images  --out_dir data/seg/val/lr_x4_real     --degradation configs/degradation_real.yaml
!python scripts/degrade.py --hr_dir data/seg/test/images --out_dir data/seg/test/lr_x4_real    --degradation configs/degradation_real.yaml
!python scripts/degrade.py --hr_dir data/seg/test/images --out_dir data/seg/test/lr_x4_bicubic --mode bicubic

## 3. Stage 1: classification

In [ ]:
!python scripts/train.py --config configs/cls_simplecnn.yaml
!python scripts/train.py --config configs/cls_dinov2.yaml

## 4. Stage 2: super-resolution

In [ ]:
!python scripts/train.py --config configs/sr_bicubic.yaml
!python scripts/train.py --config configs/sr_espcn.yaml

In [ ]:
# Fine-tuning, about 1-3 h each on a T4. Lower train.iters for a first try.
!python scripts/train.py --config configs/sr_realesrgan.yaml --opts train.iters=10000
!python scripts/train.py --config configs/sr_swinir.yaml --opts train.iters=10000

## 5. Stage 3: segmentation

In [ ]:
!python scripts/train.py --config configs/seg_unet.yaml
!python scripts/train.py --config configs/seg_segformer.yaml

## 6. LR → SR → segmentation (main table)

In [ ]:
!python scripts/pipeline_eval.py --config configs/pipeline_x4_real.yaml
!python scripts/make_table.py results/pipeline_x4_real.csv --cols method psnr ssim iou f1 tol2_f1 halluc_rate erased_rate

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/vis/pipeline_x4_real/*/*.png'))[:6]:
    print(p); display(Image(p, width=700))

## 7. Efficiency

In [ ]:
!python scripts/benchmark_efficiency.py --configs configs/sr_bicubic.yaml configs/sr_espcn.yaml configs/sr_realesrgan.yaml configs/sr_swinir.yaml --size 128 --out results/efficiency_sr.csv

In [ ]:
# Save result tables to Drive
!mkdir -p $RUNS/results && cp -r results/*.csv results/*.json $RUNS/results/ 2>/dev/null; ls $RUNS/results